# COLETA DE DIVIDENDOS

In [26]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO
import pandas as pd

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

lista = ['petr4', 'wege3', 'vale3', 'bbse3']

for i in lista:
    print(f'\nColetando Dividendos {i}')

    url = f"https://www.dadosdemercado.com.br/acoes/{i}/dividendos"
    driver.get(url)

    try:
        # ============================================================
        # TABELA 1
        # ============================================================
        tabela = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.XPATH, "/html/body/main/div/div[2]/table")
            )
        )

        html = tabela.get_attribute("outerHTML")

        df = pd.read_html(
            StringIO(html),
            decimal=',',
            thousands='.'
        )[0]

        df["Valor"] = (
            df["Valor"]
            .astype(str)
            .str.replace("*", "", regex=False)
            .str.strip()
        )

        df["Valor"] = pd.to_numeric(
            df["Valor"].str.replace(",", ".", regex=False),
            errors="coerce"
        )

        for coluna in ["Registro", "Ex", "Pagamento"]:
            df[coluna] = pd.to_datetime(
                df[coluna],
                format="%d/%m/%Y",
                errors="coerce"
            )

        print("\nTABELA 1")
        print(df)

        # ============================================================
        # TABELA 2
        # ============================================================
        tabela2 = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.XPATH, "/html/body/main/div/div[4]/table")
            )
        )

        html2 = tabela2.get_attribute("outerHTML")

        df2 = pd.read_html(
            StringIO(html2),
            decimal=',',
            thousands='.'
        )[0]

        df2["Dividendos"] = pd.to_numeric(df2["Dividendos"], errors="coerce")
        df2["Cotação"] = pd.to_numeric(
            df2["Cotação"]
            .astype(str)
            .str.replace("R$", "", regex=False)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
            .str.replace("\u202f", "", regex=False)
            .str.strip(),
            errors="coerce"
        )
        
        df2["DY"] = pd.to_numeric(
            df2["DY"]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", ".", regex=False)
            .str.strip(),
            errors="coerce"
        )/100
        
        print("\nTABELA 2")
        print(df2)

    except Exception as e:
        print("Erro:", e)

driver.quit()


Coletando Dividendos petr4

TABELA 1
          Tipo     Valor   Registro         Ex  Pagamento
0          JCP  0.202504 2026-08-21 2026-08-24 2026-12-21
1          JCP  0.674071 2026-08-21 2026-08-24 2026-11-23
2    Dividendo  0.471567 2026-08-21 2026-08-24 2026-12-21
3          JCP  0.350486 2026-06-01 2026-06-02 2026-08-20
4          JCP  0.350486 2026-06-01 2026-06-02 2026-09-21
..         ...       ...        ...        ...        ...
137        JCP  0.510000 2001-06-29 2001-07-02 2001-08-22
138  Dividendo  0.302500 2001-03-23 2001-03-26 2001-05-04
139        JCP  0.852500 2001-03-23 2001-03-26 2001-05-04
140  Dividendo  1.867500 2000-03-23 2000-03-24 2000-05-19
141        JCP  2.212500 2000-03-23 2000-03-24 2000-05-19

[142 rows x 5 columns]

TABELA 2
     Ano  Dividendos  Cotação      DY
0   2026    2.712219    44.30  0.0612
1   2025    3.295595    30.82  0.1069
2   2024    8.343873    36.19  0.2306
3   2023    7.328320    37.24  0.1968
4   2022   16.778586    24.50  0.6848
5   